In [1]:
import os
from pathlib import Path

import pandas as pd

from shadowroutes.config import settings
from shadowroutes.model.abm import Model

src_path = Path('../')
os.chdir(src_path)

## Parameters

In [2]:
# Paths
env_panel_path = Path(settings.DATA_ROOT / 'processed' / 'environment_panel.csv')
crime_panel_path = Path(settings.DATA_ROOT / 'processed' / 'crime_panel.csv')
adjacency_nodes_path = Path(settings.DATA_ROOT / 'processed' / 'adjacency_nodes.csv')

## Data

In [3]:
env_panel = pd.read_csv(env_panel_path, encoding='utf-8', dtype={"muni_id": str})
print(f'File read: {print(env_panel_path.name)}')

print(env_panel.shape)
env_panel.head()

environment_panel.csv
File read: None
(323, 11)


,muni_id,state_id,municipality,geometry,year,state,pop_total,prot_idx,mining_idx,collusion_idx,node_id
0,12033,12,Huamuxtitlán,"POLYGON ((-98.4724366200569 17.97081319957666,...",2025,Guerrero,18207,0.342432,0.000000,0.592026,12033
1,12031,12,General Canuto A. Neri,POLYGON ((-100.0758836904194 18.54482622990908...,2025,Guerrero,6913,0.103911,0.698605,0.592026,12031
2,12083,12,Ñuu Savi,POLYGON ((-98.97828861090657 17.01672138920664...,2025,Guerrero,11563,0.000000,0.000000,0.592026,12083
3,12082,12,Las Vigas,POLYGON ((-99.23351249962433 16.82400111020028...,2025,Guerrero,10136,0.000000,0.000000,0.592026,12082
4,12073,12,Zirándaro,POLYGON ((-101.2816862298956 18.54467114962441...,2025,Guerrero,18026,0.138348,0.826372,0.592026,12073


In [4]:
env_panel.dtypes

muni_id           object
state_id           int64
municipality      object
geometry          object
year               int64
state             object
pop_total          int64
prot_idx         float64
mining_idx       float64
collusion_idx    float64
node_id            int64
dtype: object

In [5]:
env_panel['muni_id'] = env_panel['muni_id'].astype('str')
env_panel['node_id'] = env_panel['node_id'].astype('str')

In [6]:
crime_panel = pd.read_csv(crime_panel_path, encoding='utf-8', dtype={"muni_id": str})
print(f'File read: {print(crime_panel_path.name)}')

print(crime_panel.shape)
crime_panel.head()

crime_panel.csv
File read: None
(41606, 12)


,year,month,state_id,state,muni_id,municipality,extortion_rate,homicide_rate,drug_dealing_rate,kidnapping_rate,human_trafficking_rate,t
0,2015,1,12,Guerrero,12001,Acapulco de Juárez,0.6,4.2,3.9,0.0,0.0,0
1,2015,1,12,Guerrero,12002,Ahuacuotzingo,0.0,0.0,0.0,0.0,0.0,0
2,2015,1,12,Guerrero,12003,Ajuchitlán del Progreso,0.0,2.6,0.0,2.6,0.0,0
3,2015,1,12,Guerrero,12004,Alcozauca de Guerrero,0.0,0.0,0.0,0.0,0.0,0
4,2015,1,12,Guerrero,12005,Alpoyeca,0.0,14.5,0.0,0.0,0.0,0


In [7]:
crime_panel.dtypes

year                        int64
month                       int64
state_id                    int64
state                      object
muni_id                    object
municipality               object
extortion_rate            float64
homicide_rate             float64
drug_dealing_rate         float64
kidnapping_rate           float64
human_trafficking_rate    float64
t                           int64
dtype: object

In [8]:
crime_panel['muni_id'] = crime_panel['muni_id'].astype('str')

In [9]:
edges = pd.read_csv(adjacency_nodes_path, encoding='utf-8', dtype={"source": str, "target": str})
print(f'File read: {print(adjacency_nodes_path.name)}')

print(edges.shape)
edges.head()

adjacency_nodes.csv
File read: None
(859, 2)


,source,target
0,12033,12066
1,12033,12005
2,12033,12024
3,12033,12045
4,12033,12070


In [10]:
edges['source'] = edges['source'].astype('str')
edges['target'] = edges['target'].astype('str')

## Execute model

#### Test crime slice

In [22]:
def get_crime_slice(crime_panel: pd.DataFrame, t: int) -> pd.DataFrame:
    """
    Return crime rates for a given time index t, indexed by muni_id.

    Parameters
    ----------
    crime_panel : pd.DataFrame
        Must contain columns:
        - t
        - muni_id
        - extortion_rate
        - homicide_rate
        - drug_dealing_rate
        - kidnapping_rate
        - human_trafficking_rate
    t : int
        Time index used in the panel.

    Returns
    -------
    slice_df : pd.DataFrame
        Indexed by muni_id with crime rate columns.
    """
    slice_df = (
        crime_panel.loc[crime_panel["t"] == t]
                   .set_index("muni_id")[[
                       "extortion_rate",
                       "homicide_rate",
                       "drug_dealing_rate",
                       "kidnapping_rate",
                       "human_trafficking_rate",
                   ]]
    )
    return slice_df

In [37]:
from shadowroutes.model.env import build_env_graph, update_env_from_crime

G = build_env_graph(env_panel, edges)
degree_hist = [d for n,d in G.degree()]

print((min(degree_hist), max(degree_hist), sum(degree_hist)/len(degree_hist)))

crime_t0 = get_crime_slice(crime_panel, t=5)
update_env_from_crime(G, crime_t0)

# Inspect one node
some_node = list(G.nodes)[0]
G.nodes[some_node]


(1, 14, 5.318885448916409)


{'muni_id': '12033',
 'state_id': 12,
 'state': 'Guerrero',
 'municipality': 'Huamuxtitlán',
 'pop_total': 18207,
 'prot_idx': 0.3424323977957196,
 'mining_idx': 0.0,
 'collusion_idx': 0.5920264574256163,
 'extortion_rate': 0.0,
 'homicide_rate': 6.3,
 'drug_dealing_rate': 0.0,
 'kidnapping_rate': 0.0,
 'human_trafficking_rate': 0.0}

In [38]:
type(list(G.nodes(data=True))[0][1]["muni_id"]), crime_t0.index.dtype


(str, dtype('O'))

In [39]:
crime_panel["t"].min(), crime_panel["t"].max(), crime_panel["t"].unique()[:5]
len(crime_t0), crime_t0.head()


(319,
          extortion_rate  homicide_rate  drug_dealing_rate  kidnapping_rate  \
 muni_id                                                                      
 12001               0.7            8.2                2.1              0.1   
 12002               0.0            0.0                0.0              0.0   
 12003               0.0            0.0                0.0              0.0   
 12004               0.0           10.0                0.0              0.0   
 12005               0.0            0.0                0.0              0.0   
 
          human_trafficking_rate  
 muni_id                          
 12001                       0.0  
 12002                       0.0  
 12003                       0.0  
 12004                       0.0  
 12005                       0.0  )

In [15]:
crime_t0.head()

,extortion_rate,homicide_rate,drug_dealing_rate,kidnapping_rate,human_trafficking_rate
muni_id,,,,,
12001,1.0,7.2,1.4,0.4,0.0
12002,0.0,0.0,0.0,0.0,0.0
12003,0.0,2.6,0.0,0.0,0.0
12004,0.0,5.0,0.0,0.0,0.0
12005,0.0,0.0,0.0,0.0,0.0


In [20]:
muni_test = G.nodes[some_node]["muni_id"]
muni_test

'12033'

In [16]:
print("index name:", crime_t0.index.name)
print("any muni_id col?", "muni_id" in crime_t0.columns)
print("is '12033' in index?", '12033' in crime_t0.index)

index name: muni_id
any muni_id col? False
is '12033' in index? True


In [24]:
'12033' in crime_t0["muni_id"].astype(str).values


KeyError: 'muni_id'

In [40]:
dff = crime_panel[crime_panel["muni_id"] == "12033"][[
    "t",
    "extortion_rate",
    "homicide_rate",
    "drug_dealing_rate",
    "kidnapping_rate",
    "human_trafficking_rate",
]]

dff

,t,extortion_rate,homicide_rate,drug_dealing_rate,kidnapping_rate,human_trafficking_rate
32,0,0.0,0.0,0.0,0.0,0.0
351,1,0.0,0.0,0.0,0.0,0.0
670,2,0.0,0.0,0.0,0.0,0.0
989,3,0.0,0.0,0.0,0.0,0.0
1308,4,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
40023,125,0.0,0.0,0.0,0.0,0.0
40346,126,0.0,0.0,0.0,0.0,0.0
40669,127,0.0,0.0,0.0,0.0,0.0
40992,128,0.0,0.0,0.0,0.0,0.0


In [41]:
crime_t5 = get_crime_slice(crime_panel, t=5)
update_env_from_crime(G, crime_t5)

# Count non-zero homicide_rate across graph
sum(1 for _, d in G.nodes(data=True) if d["homicide_rate"] > 0)

108

In [42]:
import numpy as np

homs = [d["homicide_rate"] for _, d in G.nodes(data=True)]
np.min(homs), np.max(homs), np.mean(homs)


(np.float64(0.0), np.float64(35.2), np.float64(1.3727554179566563))

In [ ]:

model = Model(
    munis_df=env_panel,
    edges_df=edges,
    crime_panel=crime_panel,
    start_t=0,
    end_t=24,          # 2 simulated years for a quick test
    n_ocg=5,
    civ_per_muni=10,
    mining_weight=1.0,
    v_escalation=0.4,
    enable_sink=True,
    seed=7
)


In [ ]:
model.run()
results = model.datacollector.get_model_vars_dataframe()
print(results.head())
